<a href="https://colab.research.google.com/github/aisyahlanim/Technical-Efficiency-Analysis/blob/main/Dea_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
try:
    from pulp import *
except:
    !pip install pulp
    from pulp import *

# 1. Load Data
df = pd.read_excel(excel_file_path, sheet_name='Sheet1')

# 2. Tentukan Input dan Output
# Pastikan tidak ada nilai 0 pada input/output agar DEA stabil
inputs = ['MonthlyIncome', 'TrainingTimesLastYear', 'TotalWorkingYears']
outputs = ['PerformanceRating', 'JobLevel', 'PercentSalaryHike']

# Filter data (ambil sampel atau bersihkan jika ada missing)
data = df[['DMU'] + inputs + outputs].copy()

# 3. Fungsi Perhitungan DEA (Input-Oriented VRS)
def solve_dea(data, inputs, outputs, orientation='input', return_to_scale='VRS'):
    dmus = data['DMU'].tolist()
    input_data = data[inputs].values.T
    output_data = data[outputs].values.T

    eff_scores = []

    for i in range(len(dmus)):
        prob = LpProblem(f"DEA_{i}", LpMinimize if orientation == 'input' else LpMaximize)

        # Variabel efisiensi
        theta = LpVariable(f"theta_{i}", lowBound=0)
        # Lambdas untuk bobot DMU
        lambdas = [LpVariable(f"lambda_{j}", lowBound=0) for j in range(len(dmus))]

        prob += theta

        # Constraints Input
        for m in range(len(inputs)):
            if orientation == 'input':
                prob += lpSum([lambdas[j] * input_data[m][j] for j in range(len(dmus))]) <= theta * input_data[m][i]
            else:
                prob += lpSum([lambdas[j] * input_data[m][j] for j in range(len(dmus))]) <= input_data[m][i]

        # Constraints Output
        for n in range(len(outputs)):
            if orientation == 'input':
                prob += lpSum([lambdas[j] * output_data[n][j] for j in range(len(dmus))]) >= output_data[n][i]
            else:
                prob += lpSum([lambdas[j] * output_data[n][j] for j in range(len(dmus))]) >= theta * output_data[n][i]

        # Add VRS constraint (Sum of Lambdas = 1)
        if return_to_scale == 'VRS':
            prob += lpSum(lambdas) == 1

        prob.solve(PULP_CBC_CMD(msg=0))
        eff_scores.append(value(theta))

    return eff_scores

# 4. Jalankan Perhitungan
print("Calculating DEA Scores (this might take a minute)...")
data['VRS_Score'] = solve_dea(data, inputs, outputs, orientation='input', return_to_scale='VRS')
data['CRS_Score'] = solve_dea(data, inputs, outputs, orientation='input', return_to_scale='CRS')
data['Scale_Efficiency'] = data['CRS_Score'] / data['VRS_Score']

# 5. Gabungkan kembali dengan data kategori untuk Tableau
final_df = pd.merge(df, data[['DMU', 'VRS_Score', 'CRS_Score', 'Scale_Efficiency']], on='DMU')

# Simpan untuk Tableau
final_df.to_excel('HR_Analytic_DEA_Results.xlsx', index=False)
print("Done! File 'HR_Analytic_DEA_Results.xlsx' siap di-download untuk Tableau.")